# BirdCLEF+ 2026 — Perch v2 Logit Direct

Perch v2 の **14,795クラス logit 出力を直接使う**アプローチ。
Embedding + 分類器ではなく、Perch の事前学習済み分類能力をそのまま活用。

## Why
- LBトップ(0.908)もPerch v2を使用。我々のLogReg(0.704)との差はlogit直接利用の可能性
- 14,795種で学習済み → 206種のLogRegより汎化性が高い
- ゼロショット種もPerch labelsに含まれていればカバー可能

## Pipeline
1. Perch labels.csv とBirdCLEFラベルのマッピングを構築（scientific_nameで照合）
2. テスト音声の5秒窓ごとにPerch logitを取得
3. マッピング済みの種のlogitをsigmoid変換して確率に

## Kaggle Settings
| Setting | Value |
|---------|-------|
| Data | `birdclef-2026` (Competition) |
| Models | `google/bird-vocalization-classifier` → `perch_v2` V1 |
| Accelerator | GPU T4 x2 |
| Internet | ON |

In [ ]:
!pip install -q "tensorflow>=2.20" perch-hoplite librosa

In [ ]:
import os, glob, pickle, warnings, time, json
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ============================================================
# Path Configuration + Audio Constants
# ============================================================
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p): COMP_DATA = p; break
assert COMP_DATA

PERCH_MODEL_PATH = None
saved_models = glob.glob('/kaggle/input/**/saved_model.pb', recursive=True)
for sm in saved_models:
    if 'perch' in sm.lower() or 'bird' in sm.lower():
        PERCH_MODEL_PATH = os.path.dirname(sm); break
if PERCH_MODEL_PATH is None and saved_models:
    PERCH_MODEL_PATH = os.path.dirname(saved_models[0])
assert PERCH_MODEL_PATH

# Audio parameters (Perch v2 の入力仕様に合わせる)
SAMPLE_RATE = 32000      # Hz: Perch v2 が期待するサンプリング周波数
DURATION = 5             # 秒: Perch v2 の入力窓長（5秒固定）
WINDOW_SAMPLES = SAMPLE_RATE * DURATION  # = 160,000 サンプル/窓

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Competition data: {COMP_DATA}')
print(f'Perch model:      {PERCH_MODEL_PATH}')
print(f'Audio: {SAMPLE_RATE}Hz, {DURATION}s window = {WINDOW_SAMPLES} samples')

In [ ]:
# ============================================================
# Load Perch v2 — embedding (1536-dim) + logits (~14,795 classes)
# ============================================================
def load_perch_full(model_path):
    """Perch v2 をロードし、(embedding, logits) を返す関数を構築する。
    
    Returns:
        predict_fn: waveform (160000,) -> (embedding (1536,), logits (14795,))
        n_perch_classes: logit の次元数
    """
    # --- perch-hoplite 経由 ---
    try:
        from perch_hoplite.zoo import model_configs
        # perch-hoplite は model_path を受け付けない場合がある
        try:
            model = model_configs.load_model_by_name('perch_v2', model_path=model_path)
        except TypeError:
            model = model_configs.load_model_by_name('perch_v2')
        
        def predict_fn(waveform):
            outputs = model.embed(waveform)
            emb = np.array(outputs.embeddings)
            if emb.ndim > 1:  # 空間embedding (5,3,1536) → 平均プーリング
                emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
            logits = np.array(outputs.logits['label']).flatten()
            return emb.astype(np.float32), logits.astype(np.float32)
        
        test_out = model.embed(np.zeros(WINDOW_SAMPLES, dtype=np.float32))
        n_labels = len(np.array(test_out.logits['label']).flatten())
        print(f'Loaded via perch-hoplite, logit classes: {n_labels}')
        return predict_fn, n_labels
    except Exception as e:
        print(f'perch-hoplite failed: {e}')
    
    # --- TF SavedModel 直接ロード ---
    model = tf.saved_model.load(model_path)
    infer_fn = model.signatures['serving_default']
    output_keys = list(infer_fn.structured_outputs.keys())
    print(f'TF output keys: {output_keys}')
    
    # 出力キーの特定: 1536次元=embedding, 5000+次元=logits
    embed_key, logit_key = None, None
    test_wf = tf.constant(np.zeros((1, WINDOW_SAMPLES), dtype=np.float32))
    test_result = infer_fn(test_wf)
    for k in output_keys:
        shape = test_result[k].shape
        print(f'  {k}: shape={shape}')
        if shape[-1] == 1536:
            embed_key = k
        elif shape[-1] > 5000:  # logits は ~14,795 クラス
            logit_key = k
    
    assert logit_key, f'Logit output not found'
    n_labels = int(test_result[logit_key].shape[-1])
    
    def predict_fn(waveform):
        wf = tf.constant(waveform[np.newaxis], dtype=tf.float32)
        result = infer_fn(wf)
        emb = result[embed_key].numpy()
        if emb.ndim > 2: emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
        elif emb.ndim == 2: emb = emb[0]
        logits = result[logit_key].numpy().flatten()
        return emb.astype(np.float32), logits.astype(np.float32)
    
    print(f'Loaded via TF, embed={embed_key}, logit={logit_key}, classes={n_labels}')
    return predict_fn, n_labels

predict_fn, n_perch_classes = load_perch_full(PERCH_MODEL_PATH)

# 動作確認: 無音5秒を入力して出力の形を確認
test_emb, test_logits = predict_fn(np.zeros(WINDOW_SAMPLES, dtype=np.float32))
print(f'\nEmbedding shape: {test_emb.shape}')    # (1536,)
print(f'Logits shape:    {test_logits.shape}')    # (14795,)
print(f'Logit range:     [{test_logits.min():.2f}, {test_logits.max():.2f}]')

In [ ]:
# ============================================================
# Perch label 名の取得
# logit の各インデックスがどの種に対応するかを特定する
# ============================================================
perch_labels = None

# 方法1: perch-hoplite の class_list から取得
try:
    from perch_hoplite.zoo import model_configs
    try:
        m = model_configs.load_model_by_name('perch_v2', model_path=PERCH_MODEL_PATH)
    except TypeError:
        m = model_configs.load_model_by_name('perch_v2')
    if hasattr(m, 'class_list') and m.class_list is not None:
        perch_labels = list(m.class_list.classes)
        print(f'Got labels from model.class_list: {len(perch_labels)}')
except Exception as e:
    print(f'Method 1 failed: {e}')

# 方法2: HuggingFace の Perch リポジトリからダウンロード
if perch_labels is None:
    try:
        !pip install -q huggingface_hub
        from huggingface_hub import hf_hub_download
        labels_path = hf_hub_download(repo_id='cgeorgiaw/Perch', filename='assets/labels.csv')
        labels_df = pd.read_csv(labels_path)
        perch_labels = labels_df.iloc[:, 0].tolist()
        print(f'Got labels from HuggingFace: {len(perch_labels)}')
    except Exception as e:
        print(f'Method 2 failed: {e}')

# 方法3: SavedModel ディレクトリ内の labels ファイルを検索
if perch_labels is None:
    for candidate in glob.glob(f'{PERCH_MODEL_PATH}/**/labels*', recursive=True):
        labels_df = pd.read_csv(candidate)
        perch_labels = labels_df.iloc[:, 0].tolist()
        print(f'Got labels from {candidate}: {len(perch_labels)}')
        break

if perch_labels:
    print(f'\nTotal Perch labels: {len(perch_labels)}')
    print(f'Sample: {perch_labels[:5]}')
else:
    print('WARNING: Could not get Perch label names')

In [ ]:
# ============================================================
# BirdCLEF species → Perch logit index マッピング構築
# taxonomy.csv の scientific_name を使って照合する
# ============================================================
tax = pd.read_csv(f'{COMP_DATA}/taxonomy.csv')
sample_sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']  # 提出要求234種

species_to_perch_idx = {}  # BirdCLEF label → Perch logit のインデックス
unmapped = []              # マッピングできなかった種

if perch_labels:
    # Perch labels を小文字化して辞書に（高速検索用）
    perch_labels_lower = {l.lower().strip(): i for i, l in enumerate(perch_labels)}
    
    for sp in species_cols:
        row = tax[tax['primary_label'].astype(str) == sp]
        if len(row) == 0:
            unmapped.append((sp, 'not in taxonomy')); continue
        
        sci_name = str(row.iloc[0]['scientific_name']).lower().strip()
        
        # 照合1: 学名の完全一致
        if sci_name in perch_labels_lower:
            species_to_perch_idx[sp] = perch_labels_lower[sci_name]; continue
        
        # 照合2: 属名+種小名（最初の2語）で一致
        base_name = ' '.join(sci_name.split()[:2])
        if base_name in perch_labels_lower:
            species_to_perch_idx[sp] = perch_labels_lower[base_name]; continue
        
        # 照合3: 英名（common_name）で一致
        common = str(row.iloc[0].get('common_name', '')).lower().strip()
        if common in perch_labels_lower:
            species_to_perch_idx[sp] = perch_labels_lower[common]; continue
        
        unmapped.append((sp, sci_name))

print(f'Mapped:   {len(species_to_perch_idx)} / {len(species_cols)} species')
print(f'Unmapped: {len(unmapped)}')
print(f'\n=== Unmapped species (Perch logit に対応なし) ===')
for sp, name in unmapped:
    print(f'  {sp}: {name}')

# マッピングを保存（提出用ノートブックで再利用）
with open(f'{OUTPUT_DIR}/species_to_perch_idx.pkl', 'wb') as f:
    pickle.dump(species_to_perch_idx, f)
print(f'\nMapping saved')

In [ ]:
# ============================================================
# 検証: 既知種の音声に対してlogitスコアが正しく高くなるか確認
# ============================================================
train_df = pd.read_csv(f'{COMP_DATA}/train.csv')

def load_waveform(path, sr=SAMPLE_RATE, duration=DURATION):
    """音声ファイルを読み込み、指定長にパディング/クロップする。"""
    try: waveform, _ = librosa.load(path, sr=sr, duration=duration)
    except: return np.zeros(sr * duration, dtype=np.float32)
    target = sr * duration
    if len(waveform) < target: waveform = np.pad(waveform, (0, target - len(waveform)))
    return waveform[:target].astype(np.float32)

print('Validation: checking logit scores for known species...')
np.random.seed(42)
test_indices = np.random.choice(len(train_df), min(20, len(train_df)), replace=False)

correct, tested = 0, 0
for idx in test_indices:
    row = train_df.iloc[idx]
    sp = str(row['primary_label'])
    if sp not in species_to_perch_idx: continue
    tested += 1
    
    path = os.path.join(COMP_DATA, 'train_audio', row['filename'])
    _, logits = predict_fn(load_waveform(path))
    
    perch_idx = species_to_perch_idx[sp]
    score = 1 / (1 + np.exp(-logits[perch_idx]))  # sigmoid で確率に変換
    rank = int((logits > logits[perch_idx]).sum()) + 1  # ランキング（1=最高）
    is_top10 = rank <= 10
    correct += is_top10
    print(f'  {sp}: sigmoid={score:.4f}, rank={rank}/{len(logits)}, top10={is_top10}', flush=True)

print(f'\nTop-10 accuracy: {correct}/{tested}')

In [ ]:
# ============================================================
# テストサウンドスケープ推論: logit を直接 sigmoid して予測
# 各5秒窓ごとに Perch logit を取得し、マッピング済みの種のスコアを出力
# ============================================================
test_dir = f'{COMP_DATA}/test_soundscapes'
test_files = sorted(f for f in os.listdir(test_dir) if f.endswith('.ogg'))
print(f'Test files: {len(test_files)}')

def sigmoid(x):
    """数値安定な sigmoid 関数。"""
    return 1 / (1 + np.exp(-np.clip(x, -50, 50)))

rows = []
t0 = time.time()
for fi, fname in enumerate(test_files):
    stem = os.path.splitext(fname)[0]
    full_wf, _ = librosa.load(os.path.join(test_dir, fname), sr=SAMPLE_RATE, mono=True)
    if len(full_wf) == 0: continue
    
    # 5秒窓で非重複スライド（Perch v2 の入力単位）
    for start in range(0, len(full_wf), WINDOW_SAMPLES):
        window = full_wf[start:start + WINDOW_SAMPLES]
        if len(window) < WINDOW_SAMPLES:  # 末尾の短い窓はゼロパディング
            window = np.pad(window, (0, WINDOW_SAMPLES - len(window)))
        
        _, logits = predict_fn(window.astype(np.float32))
        probs = sigmoid(logits)  # logit → [0, 1] の確率に変換
        
        # row_id: {ファイル名}_{バケット開始秒数}
        bucket_sec = (start // WINDOW_SAMPLES) * DURATION
        row = {'row_id': f'{stem}_{bucket_sec}'}
        for sp in species_cols:
            if sp in species_to_perch_idx:
                row[sp] = float(probs[species_to_perch_idx[sp]])
            else:
                row[sp] = 0.0  # Perch labels にない種は 0
        rows.append(row)
    
    if (fi + 1) % 50 == 0:
        print(f'  [{fi+1}/{len(test_files)}] {int(time.time()-t0)}s', flush=True)

print(f'Predictions: {len(rows)}, elapsed: {int(time.time()-t0)}s')

In [ ]:
# ============================================================
# 提出ファイル作成 + 検証
# ============================================================
submission = pd.DataFrame(rows, columns=['row_id'] + species_cols)
submission = submission.set_index('row_id').reindex(sample_sub['row_id'], fill_value=0.0).reset_index()
submission.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)

assert len(submission) == len(sample_sub)
assert list(submission.columns) == list(sample_sub.columns)
vals = submission.drop('row_id', axis=1).values
print(f'Submission saved')
print(f'  Rows:             {len(submission)}')
print(f'  Mean prediction:  {vals.mean():.6f}')
print(f'  Max prediction:   {vals.max():.4f}')
print(f'  Zero rate:        {(vals == 0).mean():.2%}')
print(f'  Non-zero species: {(vals.max(axis=0) > 0).sum()} / {len(species_cols)}')

In [ ]:
# ============================================================
# TFLite 変換（提出用ノートブック向け）
# TFLite にすることで TF バージョン非依存 + CPU 推論高速化
# ============================================================
print(f'Converting SavedModel to TFLite: {PERCH_MODEL_PATH}')
converter = tf.lite.TFLiteConverter.from_saved_model(PERCH_MODEL_PATH)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # 量子化でサイズ削減
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,   # TF ops のフォールバック
]
tflite_model = converter.convert()
tflite_path = f'{OUTPUT_DIR}/perch_v2.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f'TFLite saved: {tflite_path} ({len(tflite_model)/1024/1024:.1f} MB)')

In [ ]:
# 出力ファイルの確認
for f in sorted(glob.glob(f'{OUTPUT_DIR}/*')):
    print(f'  {os.path.basename(f):40s} {os.path.getsize(f)/1024:.1f} KB')